# 🎙️ VoiceNO! — Google Colab Free GPU Calibration

**Tagline**: *You have the right to remain silent.*

This notebook enables fast fine-tuning of **VoiceNO!** using a free Cloud GPU (e.g. Nvidia T4 on Google Colab). It solves the hardware barrier of running heavy Conformer backpropagation on low-power ultraportable laptops.

### Workflow:
1. **Record calibration videos** locally using VoiceNO! web UI on your laptop.
2. **Export Calibration Package (.zip)** from the VoiceNO! UI or CLI (`voiceno export-calib`).
3. **Upload the zip here** and run this notebook (takes ~30-60 seconds on free T4 GPU).
4. **Download `lora.pt`** back to your laptop and place it in `user_profiles/<username>/lora.pt`.
5. Switch VoiceNO! on your laptop to **Personalized Mode** — runs 100% offline and privately!

In [ ]:
# 1. Verify GPU availability
!nvidia-smi

In [ ]:
# 2. Clone VoiceNO repository and install dependencies
!git clone https://github.com/keepsloading/VoiceNO.git /content/VoiceNO || true
%cd /content/VoiceNO
!pip install -q -r requirements.txt

# Download base Auto-AVSR pretrained checkpoint (3,291-hour base model, ~956MB)
!mkdir -p checkpoints
!gdown 1r1kx7l9sWnDOCnaFHIGvOtzuhFyFA88_ -O checkpoints/vsr_trlrs2lrs3vox2avsp_base.pth


In [ ]:
# 3. Upload your exported calibration_package.zip from your laptop
from google.colab import files
import os

print("Please select your exported calibration zip file (e.g. default_calibration_package.zip):")
uploaded = files.upload()
zip_filename = list(uploaded.keys())[0]
print(f"Uploaded: {zip_filename}")

In [ ]:
# 4. Run Cloud LoRA Fine-Tuning
!python experiments/colab_train.py \
    --zip "{zip_filename}" \
    --user default \
    --epochs 15 \
    --strategy conformer_lora

In [ ]:
# 5. Download the trained personalization weights back to your laptop
from google.colab import files

result_zip = "default_personalized_weights.zip"
if os.path.exists(result_zip):
    print(f"Downloading {result_zip} to your computer...")
    files.download(result_zip)
else:
    print("File not found. Check training step logs above.")